In [7]:
#!/usr/bin/env python3
import os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm

from cnet_dataset import HSIControlNetDataset
from diffusers import DDIMScheduler, UNet2DConditionModel
from transformers import AutoTokenizer, CLIPTextModel
from warp_and_controlnet_model import WarpAndControlNetModel
from model_with_camera import NewUNet2DConditionModel
from peft import LoraConfig
from safetensors.torch import load_file

# ---------------- Config ----------------
ROOT_DIR     = "/data/train/train_30/"
CNET_CKPT    = "/data/nd_model_7_lora/checkpoint-12000/controlnet"
UNET_WEIGHTS = "/home/viplab/liukejia/project/depth_model/nd_ckpt_diffuser/nd_mv_ema_diffuser.pth"
LORA_CKPT  = "/data/nd_model_7_lora/checkpoint-12000/unet"
BASE_MODEL   = "stable-diffusion-v1-5/stable-diffusion-v1-5"
COMPARE_ROOT = "./test12000_7"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32  # 修正：CPU 不用 fp16

# Must match training
NUM_FRAMES   = 4
LATENT_SIZE  = 64
# COND_SIZE    = 256
COND_SIZE    = LATENT_SIZE * 8  # 64 -> 512，和 SD/ControlNet 对齐
HSI_CHANNELS = 48
CAMERA_DIM   = 16

# 尝试多个 CFG
CFG_SCALES = [1,2,4,6,10]
time_step = 50

os.makedirs(COMPARE_ROOT, exist_ok=True)


In [8]:

# ---------------- 1. Load components ----------------
print("Loading models...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    BASE_MODEL, subfolder="text_encoder", torch_dtype=DTYPE
).to(DEVICE).eval()

controlnet = WarpAndControlNetModel.from_pretrained(CNET_CKPT, torch_dtype=DTYPE).to(DEVICE).eval()
if hasattr(controlnet, "warper") and DEVICE == "cuda" and DTYPE == torch.float16:
    controlnet.warper = controlnet.warper.half()

temp_model = UNet2DConditionModel.from_pretrained(BASE_MODEL, subfolder="unet", low_cpu_mem_usage=True)
unet = NewUNet2DConditionModel(**temp_model.config, camera_dim=CAMERA_DIM)

# 1) 先加载基础 UNet 权重
unet_state = torch.load(UNET_WEIGHTS, map_location="cpu")
if isinstance(unet_state, dict) and "state_dict" in unet_state:
    unet_state = unet_state["state_dict"]
unet.load_state_dict(unet_state, strict=True)
print("Loaded base UNet weights")

# 2) 加载增量权重（LoRA 或 full unet safetensors）
loaded_lora = False

# A) PEFT adapter 目录
if os.path.exists(os.path.join(LORA_CKPT, "config.json")):
    try:
        unet_lora_config = LoraConfig(
            r=8,
            lora_alpha=8,
            init_lora_weights="gaussian",
            target_modules=["to_k_proj", "to_q_proj", "to_v_proj", "to_out.0"],
        )
        unet.add_adapter(unet_lora_config, adapter_name="unet_lora")
        unet.load_adapter(LORA_CKPT, adapter_name="unet_lora")
        if hasattr(unet, "set_adapter"):
            unet.set_adapter("unet_lora")
        elif hasattr(unet, "set_adapters"):
            unet.set_adapters(["unet_lora"])
        loaded_lora = True
        print(f"Loaded LoRA by PEFT from: {LORA_CKPT}")
    except Exception as e:
        print(f"[warn] PEFT load_adapter failed: {e}")

# B) diffusers LoRA 文件名
if not loaded_lora:
    for wname in ["pytorch_lora_weights.safetensors", "pytorch_lora_weights.bin"]:
        if os.path.exists(os.path.join(LORA_CKPT, wname)):
            try:
                unet.load_attn_procs(LORA_CKPT, weight_name=wname)
                loaded_lora = True
                print(f"Loaded LoRA by diffusers attn_procs: {wname}")
                break
            except Exception as e:
                print(f"[warn] load_attn_procs({wname}) failed: {e}")

# C) 兜底：目录里是 full unet 权重（你的情况）
if not loaded_lora:
    full_unet_st = os.path.join(LORA_CKPT, "diffusion_pytorch_model.safetensors")
    if os.path.exists(full_unet_st):
        sd = load_file(full_unet_st)
        missing, unexpected = unet.load_state_dict(sd, strict=False)
        loaded_lora = True
        print(f"[fallback] Loaded FULL UNet weights from: {full_unet_st}")
        print(f"[fallback] missing={len(missing)}, unexpected={len(unexpected)}")
    else:
        raise RuntimeError(
            f"LoRA/增量权重加载失败：{LORA_CKPT}\n"
            f"未找到 adapter_config.json / pytorch_lora_weights.* / diffusion_pytorch_model.safetensors"
        )

unet = unet.to(DEVICE, dtype=DTYPE).eval()

scheduler = DDIMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")
scheduler.set_timesteps(time_step)

# ---------------- 2. Dataset ----------------
dataset = HSIControlNetDataset(root_dir=ROOT_DIR, image_size=LATENT_SIZE)

def _to_tensor(x):
    if isinstance(x, torch.Tensor):
        return x.to(DEVICE, dtype=DTYPE)
    return torch.from_numpy(np.asarray(x)).to(DEVICE, dtype=DTYPE)

def _prompt_embeds(prompt: str):
    text_input = tokenizer(
        prompt, padding="max_length", max_length=tokenizer.model_max_length,
        truncation=True, return_tensors="pt",
    )
    uncond_input = tokenizer(
        "", padding="max_length", max_length=tokenizer.model_max_length,
        truncation=True, return_tensors="pt",
    )
    cond = text_encoder(text_input.input_ids.to(DEVICE))[0]
    uncond = text_encoder(uncond_input.input_ids.to(DEVICE))[0]
    cond = cond.repeat_interleave(NUM_FRAMES, dim=0)
    uncond = uncond.repeat_interleave(NUM_FRAMES, dim=0)
    return cond, uncond



Loading models...
Warper weights loaded from /data/nd_model_7_lora/checkpoint-12000/controlnet/warper.pt
Loaded base UNet weights
[warn] PEFT load_adapter failed: 'NewUNet2DConditionModel' object has no attribute 'load_adapter'
[fallback] Loaded FULL UNet weights from: /data/nd_model_7_lora/checkpoint-12000/unet/diffusion_pytorch_model.safetensors
[fallback] missing=0, unexpected=0
dataset name: /data/train/train_30/
json collected
folders collected
folders len 48000
groups len 12000


In [9]:
# ---------------- 3. Inference ----------------
@torch.no_grad()
def infer(sample, idx, guidance_scale=7.0, seed=42):
    if hasattr(dataset, "groups"):
        print(f"[debug] idx={idx} folders={dataset.groups[idx]}")

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    prompt = sample["text"]
    cond_embed, uncond_embed = _prompt_embeds(prompt)

    # 与 dataset 一致：conditioning_image 是 (F, 48, H, W) 的 HSI 立方体
    cond_image = _to_tensor(sample["conditioning_image"])
    cond_image = torch.nn.functional.interpolate(
        cond_image, size=(COND_SIZE, COND_SIZE), mode="bilinear", align_corners=False
    )
    assert cond_image.shape[0] == NUM_FRAMES and cond_image.shape[1] == HSI_CHANNELS, \
        f"Unexpected cond_image shape {cond_image.shape}"

    camera_matrix = _to_tensor(sample["camera_matrix"])
    assert camera_matrix.shape == (NUM_FRAMES, CAMERA_DIM), \
        f"Unexpected camera_matrix shape {camera_matrix.shape}"

    latents = torch.randn(
        (NUM_FRAMES, 4, LATENT_SIZE, LATENT_SIZE),
        generator=generator, device=DEVICE, dtype=DTYPE
    )
    latents = latents * scheduler.init_noise_sigma

    print(f"Running inference: {prompt} | cfg={guidance_scale}")

    for t in tqdm(scheduler.timesteps):
        if guidance_scale > 1.0:
            latent_model_input = torch.cat([latents, latents], dim=0)
            prompt_embeds_input = torch.cat([uncond_embed, cond_embed], dim=0)
            cond_image_input = torch.cat([cond_image, cond_image], dim=0)
            camera_matrix_input = torch.cat([camera_matrix, camera_matrix], dim=0)
        else:
            latent_model_input = latents
            prompt_embeds_input = cond_embed
            cond_image_input = cond_image
            camera_matrix_input = camera_matrix

        # 修正：DDIM 需要 scale_model_input
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        down_block_res_samples, mid_block_res_sample = controlnet(
            sample=latent_model_input,
            timestep=t,
            encoder_hidden_states=prompt_embeds_input,
            controlnet_cond=cond_image_input,
            return_dict=False,
        )

        noise_pred = unet(
            sample=latent_model_input,
            timestep=t,
            encoder_hidden_states=prompt_embeds_input,
            down_block_additional_residuals=down_block_res_samples,
            mid_block_additional_residual=mid_block_res_sample,
            camera=camera_matrix_input,
        ).sample

        if guidance_scale > 1.0:
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # 可视化（latent 近似）
    image = (latents.float() / 2 + 0.5).clamp(0, 1)
    image = torch.nn.functional.interpolate(image, size=(256, 256), mode="bilinear", align_corners=False)
    image = image.cpu().permute(0, 2, 3, 1).numpy()  # (F,256,256,4)

    # 原图
    raw = np.array(sample["image"]) if "image" in sample else None
    raw_canvas = Image.new("RGB", (256 * NUM_FRAMES, 256))
    raw_ch4_canvas = Image.new("RGB", (256 * NUM_FRAMES, 256))

    if raw is not None:
        raw = raw[:, :4, ...].transpose(0, 2, 3, 1)  # (F,H,W,4)
        for i in range(min(NUM_FRAMES, raw.shape[0])):
            # 第1行：原图RGB
            x_rgb = raw[i, ..., :3]
            x_rgb = (x_rgb - x_rgb.min()) / (x_rgb.max() - x_rgb.min() + 1e-8)
            raw_rgb_np = (x_rgb * 255).astype(np.uint8)
            raw_img = Image.fromarray(raw_rgb_np).resize((256, 256), Image.BILINEAR)
            raw_canvas.paste(raw_img, (256 * i, 0))

            # 第4行：原图第4维（通道索引3）
            x_c4 = raw[i, ..., 3]
            x_c4 = (x_c4 - x_c4.min()) / (x_c4.max() - x_c4.min() + 1e-8)
            x_c4_u8 = (x_c4 * 255).astype(np.uint8)
            x_c4_rgb = np.repeat(x_c4_u8[..., None], 3, axis=-1)
            raw_ch4_canvas.paste(Image.fromarray(x_c4_rgb).resize((256, 256), Image.BILINEAR), (256 * i, 0))

    # 第2行：按 dataset cond 处理方式可视化 HSI 条件（48 通道 -> 伪彩色）
    cond_canvas = Image.new("RGB", (256 * NUM_FRAMES, 256))
    cond_vis = cond_image.float().cpu().numpy()  # (F,48,H,W)
    cond_bands = [10, 24, 40]  # 选 3 个波段做伪彩色，可按需调整
    cond_bands = [min(b, HSI_CHANNELS - 1) for b in cond_bands]
    for i in range(NUM_FRAMES):
        cond_rgb = np.stack([cond_vis[i, cond_bands[0]], cond_vis[i, cond_bands[1]], cond_vis[i, cond_bands[2]]], axis=-1)
        cond_rgb = np.clip(cond_rgb, 0.0, 1.0)
        cond_rgb = np.power(cond_rgb, 1.0 / 2.2)  # 轻微 gamma，避免发灰/发白
        cond_rgb = (cond_rgb * 255).astype(np.uint8)
        cond_canvas.paste(Image.fromarray(cond_rgb).resize((256, 256), Image.BILINEAR), (256 * i, 0))

    # 第3行：预测 RGB（逐帧 min-max，避免直接 /2+0.5 导致偏白）
    rgb_canvas = Image.new("RGB", (256 * NUM_FRAMES, 256))
    for i in range(NUM_FRAMES):
        rgb = image[i, ..., :3]
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
        rgb_np = (rgb * 255).astype(np.uint8)
        rgb_canvas.paste(Image.fromarray(rgb_np), (256 * i, 0))

    # 第5行：预测最后一通道灰度
    pred_ch_canvas = Image.new("RGB", (256 * NUM_FRAMES, 256))
    for i in range(NUM_FRAMES):
        ch = image[i, ..., -1]
        ch = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)
        ch_u8 = (ch * 255).astype(np.uint8)
        ch_rgb = np.repeat(ch_u8[..., None], 3, axis=-1)
        pred_ch_canvas.paste(Image.fromarray(ch_rgb), (256 * i, 0))

    # 5行拼接：原图RGB / 条件HSI伪彩色 / 预测RGB / 原图第4维 / 预测第4维
    grid = Image.new("RGB", (256 * NUM_FRAMES, 256 * 5))
    grid.paste(raw_canvas,      (0, 0))
    grid.paste(cond_canvas,     (0, 256))
    grid.paste(rgb_canvas,      (0, 512))
    grid.paste(raw_ch4_canvas,  (0, 768))
    grid.paste(pred_ch_canvas,  (0, 1024))
    return grid

# ---------------- 4. Run a few samples (multi-CFG) ----------------
for cfg in CFG_SCALES:
    cfg_dir = os.path.join(COMPARE_ROOT, f"cfg_{cfg:g}")
    os.makedirs(cfg_dir, exist_ok=True)
    print(f"\n===== Running CFG={cfg}, save to {cfg_dir} =====")

    for idx in range(3):
        sample = dataset[idx]
        result_img = infer(sample, idx=idx, guidance_scale=cfg, seed=42 + idx)
        save_path = os.path.join(cfg_dir, f"result_{idx}.png")
        result_img.save(save_path)
        print(f"Saved: {save_path}")

print("Done!")


===== Running CFG=1, save to ./test12000_7/cfg_1 =====
[debug] idx=0 folders=['/data/train/train_30/2025-08-19_19-35-52', '/data/train/train_30/2025-08-19_19-35-53', '/data/train/train_30/2025-08-19_19-35-54', '/data/train/train_30/2025-08-19_19-35-55']
Running inference: city topview from camera, camera conf as following. {camera_height:100.0,tilt_deg:30.0,lens:100.0} | cfg=1


100%|██████████| 50/50 [00:06<00:00,  7.94it/s]


Saved: ./test12000_7/cfg_1/result_0.png
[debug] idx=1 folders=['/data/train/train_30/2025-08-20_20-12-41', '/data/train/train_30/2025-08-20_20-12-42', '/data/train/train_30/2025-08-20_20-12-43', '/data/train/train_30/2025-08-20_20-12-44']
Running inference: city topview from camera, camera conf as following. {camera_height:100.0,tilt_deg:30.0,lens:100.0} | cfg=1


100%|██████████| 50/50 [00:06<00:00,  7.67it/s]


Saved: ./test12000_7/cfg_1/result_1.png
[debug] idx=2 folders=['/data/train/train_30/2025-08-20_20-12-45', '/data/train/train_30/2025-08-20_20-12-46', '/data/train/train_30/2025-08-20_20-12-46_2', '/data/train/train_30/2025-08-20_20-12-47']
Running inference: city topview from camera, camera conf as following. {camera_height:100.0,tilt_deg:30.0,lens:100.0} | cfg=1


...

Saved: ./test12000_7/cfg_10/result_0.png
[debug] idx=1 folders=['/data/train/train_30/2025-08-20_20-12-41', '/data/train/train_30/2025-08-20_20-12-42', '/data/train/train_30/2025-08-20_20-12-43', '/data/train/train_30/2025-08-20_20-12-44']
Running inference: city topview from camera, camera conf as following. {camera_height:100.0,tilt_deg:30.0,lens:100.0} | cfg=10


100%|██████████| 50/50 [00:15<00:00,  3.29it/s]


Saved: ./test12000_7/cfg_10/result_1.png
[debug] idx=2 folders=['/data/train/train_30/2025-08-20_20-12-45', '/data/train/train_30/2025-08-20_20-12-46', '/data/train/train_30/2025-08-20_20-12-46_2', '/data/train/train_30/2025-08-20_20-12-47']
Running inference: city topview from camera, camera conf as following. {camera_height:100.0,tilt_deg:30.0,lens:100.0} | cfg=10


100%|██████████| 50/50 [00:12<00:00,  3.86it/s]


Saved: ./test12000_7/cfg_10/result_2.png
Done!
